# 🎬 Netflix Content Analysis — Exploratory Data Analysis

**Dataset:** Netflix Movies and TV Shows | [Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**Author:** Your Name  
**Tools:** Python · Pandas · Matplotlib · Seaborn · SQLite

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Data Cleaning](#2)
3. [Summary Statistics](#3)
4. [Movies vs TV Shows](#4)
5. [Content Trends Over Time](#5)
6. [Genre Analysis](#6)
7. [Country Distribution](#7)
8. [Rating Analysis](#8)
9. [Director Insights](#9)
10. [Duration Analysis](#10)
11. [SQL Analysis](#11)
12. [Key Findings](#12)

## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
})
PALETTE = {'Movie': '#E50914', 'TV Show': '#F5822A'}
print('✅ Libraries loaded')

In [ ]:
df = pd.read_csv('../data/netflix_titles.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing': missing, 'Percentage': missing_pct}).query('Missing > 0')

## 2. Data Cleaning <a id='2'></a>

In [ ]:
# Fill nulls
df['director'] = df['director'].fillna('Unknown')
df['cast']     = df['cast'].fillna('Unknown')
df['country']  = df['country'].fillna('Unknown')
df['rating']   = df['rating'].fillna('Not Rated')

# Drop rows with missing date_added
df = df.dropna(subset=['date_added'])

# Parse date
df['date_added']  = pd.to_datetime(df['date_added'].str.strip())
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()
df['month_num']   = df['date_added'].dt.month

# Duration
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)

print(f'✅ Cleaned. Shape: {df.shape}')
df.head()

## 3. Summary Statistics <a id='3'></a>

In [ ]:
print('='*50)
print(f"  Total titles   : {len(df):,}")
print(f"  Movies         : {(df['type']=='Movie').sum():,}")
print(f"  TV Shows       : {(df['type']=='TV Show').sum():,}")
print(f"  Countries      : {df['country'].nunique():,}")
print(f"  Release range  : {int(df['release_year'].min())} – {int(df['release_year'].max())}")
movies = df[df['type']=='Movie']
print(f"  Avg movie dur. : {movies['duration_value'].mean():.0f} min")
print(f"  Top rating     : {df['rating'].value_counts().idxmax()}")
print(f"  Top genre      : {df['listed_in'].str.split(', ').explode().value_counts().idxmax()}")
print('='*50)

## 4. Movies vs TV Shows <a id='4'></a>

In [ ]:
counts = df['type'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=[PALETTE.get(t,'#888') for t in counts.index],
    startangle=90, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(13); at.set_fontweight('bold'); at.set_color('white')

ax.text(0, 0, f'{len(df):,}\nTitles', ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_title('Movies vs TV Shows on Netflix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/01_content_type_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Content Trends Over Time <a id='5'></a>

In [ ]:
yearly = (
    df[df['year_added'].between(2015, 2021)]
    .groupby(['year_added', 'type'])
    .size().unstack(fill_value=0).reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(yearly))
ax.bar(x, yearly['Movie'],   color='#E50914', label='Movies',   width=0.55)
ax.bar(x, yearly['TV Show'], color='#F5822A', label='TV Shows', width=0.55,
       bottom=yearly['Movie'])

ax.set_xticks(list(x))
ax.set_xticklabels(yearly['year_added'].astype(int).tolist())
ax.set_xlabel('Year added'); ax.set_ylabel('Number of titles')
ax.set_title('Content Added per Year (2015–2021)', fontsize=14, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../visualizations/02_yearly_additions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Genre Analysis <a id='6'></a>

In [ ]:
genres = df['listed_in'].str.split(', ').explode().str.strip().value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [plt.cm.Reds_r(i / len(genres)) for i in range(len(genres))]
ax.barh(genres.index[::-1], genres.values[::-1], color=colors[::-1], height=0.65)

for i, (idx, val) in enumerate(zip(genres.index[::-1], genres.values[::-1])):
    ax.text(val + 20, i, f'{val:,}', va='center', fontsize=10)

ax.set_xlabel('Number of titles')
ax.set_title('Top 10 Genres on Netflix', fontsize=14, fontweight='bold')
ax.set_xlim(0, genres.max() * 1.18)
plt.tight_layout()
plt.savefig('../visualizations/03_top_genres.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Country Distribution <a id='7'></a>

In [ ]:
countries = (
    df[df['country'] != 'Unknown']['country']
    .str.split(', ').explode().str.strip()
    .value_counts().head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(countries.index[::-1], countries.values[::-1], color='#378ADD', height=0.65)
for i, val in enumerate(countries.values[::-1]):
    ax.text(val + 30, i, f'{val:,}', va='center', fontsize=10)

ax.set_xlabel('Number of titles')
ax.set_title('Top 10 Content-Producing Countries', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/04_top_countries.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Rating Analysis <a id='8'></a>

In [ ]:
valid_ratings = ['G','PG','PG-13','R','TV-Y','TV-Y7','TV-G','TV-PG','TV-14','TV-MA','NR']
ratings = df[df['rating'].isin(valid_ratings)]['rating'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ['#E50914' if r == ratings.idxmax() else '#F5822A' for r in ratings.index]
ax.bar(ratings.index, ratings.values, color=bar_colors, width=0.6, edgecolor='white')

for i, val in enumerate(ratings.values):
    ax.text(i, val + 30, str(val), ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Content Rating'); ax.set_ylabel('Titles')
ax.set_title('Content Rating Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/05_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Director Insights <a id='9'></a>

In [ ]:
directors = (
    df[df['director'] != 'Unknown']['director']
    .str.split(', ').explode().str.strip()
    .value_counts().head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(directors.index[::-1], directors.values[::-1], color='#6C5CE7', height=0.65)
for i, val in enumerate(directors.values[::-1]):
    ax.text(val + 0.3, i, str(val), va='center', fontsize=10)

ax.set_xlabel('Number of titles directed')
ax.set_title('Top 10 Directors on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/07_top_directors.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Duration Analysis <a id='10'></a>

In [ ]:
movies = df[(df['type'] == 'Movie') & df['duration_value'].notna()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(movies['duration_value'], bins=40, color='#E50914', edgecolor='white', alpha=0.85)
ax.axvline(movies['duration_value'].median(), color='#333', linestyle='--', linewidth=1.5,
           label=f"Median: {int(movies['duration_value'].median())} min")
ax.set_xlabel('Duration (minutes)'); ax.set_ylabel('Number of movies')
ax.set_title('Movie Duration Distribution', fontsize=14, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../visualizations/08_movie_duration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMovie Duration Stats:")
print(movies['duration_value'].describe())

## 11. SQL Analysis <a id='11'></a>

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('netflix', conn, index=False, if_exists='replace')

# Top 10 countries
q = '''
    SELECT country, COUNT(*) AS titles
    FROM netflix
    WHERE country != 'Unknown'
    GROUP BY country
    ORDER BY titles DESC
    LIMIT 10
'''
pd.read_sql_query(q, conn)

In [ ]:
# Year-over-year growth
q2 = '''
    SELECT year_added, COUNT(*) AS cnt
    FROM netflix
    WHERE year_added BETWEEN 2015 AND 2021
    GROUP BY year_added
    ORDER BY year_added
'''
pd.read_sql_query(q2, conn)

In [ ]:
# Top directors
q3 = '''
    SELECT director, COUNT(*) AS titles, GROUP_CONCAT(DISTINCT type) AS types
    FROM netflix
    WHERE director != 'Unknown'
    GROUP BY director
    ORDER BY titles DESC
    LIMIT 10
'''
pd.read_sql_query(q3, conn)

## 12. Key Findings <a id='12'></a>

| # | Finding |
|---|---|
| 1 | **Movies dominate** — 69.6% of the library |
| 2 | **Peak growth in 2019** — content additions hit an all-time high |
| 3 | **Dramas are the most popular genre** across both Movies and TV Shows |
| 4 | **USA is the top producer** with ~42% of all titles |
| 5 | **TV-MA is the most common rating** — Netflix skews toward adult content |
| 6 | **India is #2** in content production, reflecting Netflix's global expansion |
| 7 | **Typical movie is ~90–100 minutes** long based on the median |
| 8 | **July is the peak month** for new content additions |

---

*Analysis by [Your Name] | Dataset: Kaggle Netflix Shows*